In [1]:
import numpy as np
import pandas as pd

In [2]:
temp_df = pd.read_csv('IMDB Dataset.csv')

In [3]:
df = temp_df.iloc[:10000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_18757/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [8]:
import re 
def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

In [9]:
df['review']=df['review'].apply(remove_html_tags)

/tmp/ipykernel_18757/2177839855.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review']=df['review'].apply(remove_html_tags)


In [10]:
df['review'] = df['review'].str.lower()

/tmp/ipykernel_18757/724319867.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].str.lower()


In [11]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

/tmp/ipykernel_18757/2779382103.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))


In [15]:
import gensim
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [17]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))


In [20]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [21]:
model.build_vocab(story)

In [22]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5849882, 6186875)

In [23]:
len(model.wv.index_to_key)

31845

In [24]:
def document_vector(doc):
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [25]:
document_vector(df['review'][0])

array([-0.0340856 ,  0.30203104,  0.11949815, -0.08978475, -0.02379045,
       -0.49325418,  0.17584717,  0.8354629 , -0.17860587, -0.05641781,
       -0.00308852, -0.556691  , -0.05117341,  0.22889459,  0.03289013,
       -0.2237677 ,  0.14329539, -0.5626605 ,  0.10989192, -0.7258185 ,
        0.14424117,  0.09442353,  0.2604108 , -0.1429285 , -0.26006386,
       -0.16804823, -0.1846979 , -0.3517797 , -0.27292743,  0.03802511,
        0.48921853,  0.09786861,  0.00471562, -0.26409835, -0.33084786,
        0.20773168,  0.05387879, -0.42001736, -0.19057564, -0.7249326 ,
        0.17541215, -0.30289406, -0.2826951 ,  0.03302402,  0.44524285,
       -0.1622263 , -0.4039603 , -0.06695728,  0.16008113,  0.22782376,
        0.13871379, -0.25883254, -0.20760047, -0.13668223, -0.40957364,
        0.04712659,  0.3491881 ,  0.11375917, -0.35771495, -0.00312049,
       -0.0686461 ,  0.13012707, -0.15768306,  0.02554128, -0.3693591 ,
        0.32706267, -0.01595982,  0.19739471, -0.6288367 ,  0.32

In [26]:
from tqdm import tqdm

In [27]:
x = []
for doc in tqdm(df['review']):
    x.append(document_vector(doc))

100%|██████████| 9983/9983 [05:48<00:00, 28.63it/s]


In [28]:
x= np.array(x)
x.shape

(9983, 100)

In [29]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment'])

In [30]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [31]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [36]:
mnb = RandomForestClassifier()
mnb.fit(x_train, y_train)
y_pred = mnb.predict(x_test)
accuracy_score(y_test, y_pred)

0.7746619929894842